In [1]:
from binance.streams import BinanceSocketManager


ModuleNotFoundError: No module named 'binance.streams'

In [3]:
pip install python-binance


Note: you may need to restart the kernel to use updated packages.


In [5]:
pip install --upgrade python-binance


   ---------------------------------------- 0.0/130.8 kB ? eta -:--:--
   --------- ----------------------------- 30.7/130.8 kB 660.6 kB/s eta 0:00:01
   ---------------------------- ----------- 92.2/130.8 kB 1.1 MB/s eta 0:00:01
   ---------------------------------------- 130.8/130.8 kB 1.3 MB/s eta 0:00:00
  Attempting uninstall: python-binance
    Found existing installation: python-binance 1.0.24
    Uninstalling python-binance-1.0.24:
      Successfully uninstalled python-binance-1.0.24
Note: you may need to restart the kernel to use updated packages.


In [ ]:
from binance.client import Client
import numpy as np
import pandas as pd
import time
import logging

# Configure logging to track errors and warnings
logging.basicConfig(filename='binance_volume_log.txt', level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# Initialize the Binance client with your API keys
api_key = "your_api_key"
api_secret = "your_api_secret"
client = Client(api_key, api_secret)

# Define the volume threshold multiplier (set as 20 for now)
volume_threshold_multiplier = 10

# Function to check if a symbol meets the volume threshold criteria
def check_volume_threshold(symbol, max_retries=5, backoff_factor=2):
    retries = 0
    while retries < max_retries:
        try:
            # Fetch the last 1400 five-minute bars (maximum limit)
            klines_5min = client.futures_klines(symbol=symbol, interval=Client.KLINE_INTERVAL_5MINUTE, limit=1400)
            df_5min = pd.DataFrame(klines_5min, columns=['timestamp', 'open', 'high', 'low', 'close', 'volume', 'close_time', 
                                                         'quote_asset_volume', 'number_of_trades', 'taker_buy_base_asset_volume', 
                                                         'taker_buy_quote_asset_volume', 'ignore'])
            
            # Convert volume to float
            df_5min['volume'] = df_5min['volume'].astype(float)

            # Calculate the standard deviation of the 5-minute volume
            std_volume_5min = np.std(df_5min['volume'])

            # Calculate the threshold based on just the standard deviation (no mean)
            volume_threshold = std_volume_5min * volume_threshold_multiplier

            # Fetch the latest 1-minute bar
            klines_1min = client.futures_klines(symbol=symbol, interval=Client.KLINE_INTERVAL_1MINUTE, limit=1)
            df_1min = pd.DataFrame(klines_1min, columns=['timestamp', 'open', 'high', 'low', 'close', 'volume', 'close_time', 
                                                         'quote_asset_volume', 'number_of_trades', 'taker_buy_base_asset_volume', 
                                                         'taker_buy_quote_asset_volume', 'ignore'])

            # Convert the 1-minute volume to float
            df_1min['volume'] = df_1min['volume'].astype(float)

            # Check if the last 1-minute bar's volume exceeds the threshold
            last_1min_bar_volume = df_1min['volume'].iloc[-1]
            if last_1min_bar_volume > volume_threshold:
                return True
            else:
                return False

        except Exception as e:
            # Handle network or rate limit errors
            if 'APIError(code=-1003)' in str(e):
                logging.warning(f"Rate limit exceeded for {symbol}. Retrying after backoff...")
                time.sleep(60)  # If rate-limited, wait for 60 seconds
            else:
                logging.error(f"Error fetching data for {symbol}: {e}. Retrying after {backoff_factor ** retries} seconds...")
                time.sleep(backoff_factor ** retries)  # Exponential backoff on other errors
                retries += 1

    logging.error(f"Failed to fetch data for {symbol} after {max_retries} retries.")
    return False

# Function to fetch all USDT-paired futures symbols
def fetch_usdt_futures_symbols():
    try:
        futures_info = client.futures_exchange_info()
        usdt_symbols = [symbol['symbol'] for symbol in futures_info['symbols'] if symbol['symbol'].endswith('USDT')]
        return usdt_symbols
    except Exception as e:
        logging.error(f"Error fetching USDT futures symbols: {e}")
        return []

# Total duration (3 hours in seconds)
total_duration = 3 * 60 * 60  # 3 hours in seconds

# Interval duration (3 minutes in seconds)
interval = 3 * 60  # 3 minutes in seconds

# Start time
start_time = time.time()

# Run the volume threshold check every 3 minutes for 3 hours
usdt_symbols = fetch_usdt_futures_symbols()

while time.time() - start_time < total_duration:
    print(f"Checking symbols at {time.strftime('%Y-%m-%d %H:%M:%S')}")

    for symbol in usdt_symbols:
        if check_volume_threshold(symbol):
            print(f"{symbol} meets the volume threshold criteria.")
    
    # Wait for the interval (3 minutes) before checking again
    time.sleep(interval)

print("The 3-hour monitoring period has ended.")

Checking symbols at 2025-06-16 11:09:21


In [ ]:
import ccxt
import time
from datetime import datetime

# Initialize the Binance futures exchange
exchange = ccxt.binance({
    'enableRateLimit': True,
    'options': {'defaultType': 'future'}
})

# Function to fetch the high and low prices of the previous day's daily candle
def get_previous_day_high_low(symbol):
    try:
        # Fetch the daily candles for the previous day
        ohlcv = exchange.fetch_ohlcv(symbol, '1d', limit=2)  # Get the last 2 days of data
        # The previous day's data will be at index 0, the most recent at index 1
        previous_day = ohlcv[0]  # (timestamp, open, high, low, close, volume)
        high = previous_day[2]  # High of the previous day
        low = previous_day[3]   # Low of the previous day
        return high, low
    except Exception as e:
        print(f"Error fetching daily data for {symbol}: {e}")
        return None, None

# Function to fetch the current price (close of 1-minute candle)
def get_current_price(symbol):
    try:
        # Fetch the most recent 1-minute candle
        ohlcv = exchange.fetch_ohlcv(symbol, '1m', limit=1)
        current_price = ohlcv[-1][4]  # Closing price of the most recent 1-minute candle
        return current_price
    except Exception as e:
        print(f"Error fetching 1-minute data for {symbol}: {e}")
        return None

# Function to check if a symbol is valid and available
def is_valid_symbol(symbol):
    try:
        # Attempt to fetch ticker data for the symbol to verify its existence
        exchange.fetch_ticker(symbol)
        return True
    except ccxt.BaseError:
        # If the symbol is invalid or unavailable, return False silently
        return False

# Function to check for breakouts or breakdowns
def check_breakouts_and_breakdowns():
    # Fetch all futures symbols for USDT pairs
    markets = exchange.load_markets()
    futures_symbols = [symbol for symbol in markets if symbol.endswith('/USDT')]

    for symbol in futures_symbols:
        # Check if the symbol is valid and available on Binance
        if not is_valid_symbol(symbol):
            continue  # Skip invalid or unavailable symbols silently

        # Fetch high/low for previous day and current price
        high, low = get_previous_day_high_low(symbol)
        current_price = get_current_price(symbol)
        
        if high is None or low is None or current_price is None:
            continue  # Skip symbols with missing data

        # Check for breakout or breakdown conditions
        if current_price > high:
            print(f"{symbol}: Breakout above yesterday's high!")
        elif current_price < low:
            print(f"{symbol}: Breakdown below yesterday's low!")
        elif current_price > low and current_price < high:
            print(f"{symbol}: Potential breakout or breakdown in progress!")

# Run the checker every minute
while True:
    check_breakouts_and_breakdowns()
    time.sleep(60)  # Wait for the next minute before checking again



BTC/USDT: Potential breakout or breakdown in progress!
ETH/USDT: Breakout above yesterday's high!
BNB/USDT: Potential breakout or breakdown in progress!
NEO/USDT: Potential breakout or breakdown in progress!
LTC/USDT: Potential breakout or breakdown in progress!
QTUM/USDT: Potential breakout or breakdown in progress!
ADA/USDT: Potential breakout or breakdown in progress!
XRP/USDT: Potential breakout or breakdown in progress!
EOS/USDT: Breakout above yesterday's high!
IOTA/USDT: Potential breakout or breakdown in progress!
XLM/USDT: Potential breakout or breakdown in progress!
ONT/USDT: Potential breakout or breakdown in progress!
TRX/USDT: Potential breakout or breakdown in progress!
ETC/USDT: Breakout above yesterday's high!
ICX/USDT: Potential breakout or breakdown in progress!
NULS/USDT: Potential breakout or breakdown in progress!
VET/USDT: Potential breakout or breakdown in progress!
BSV/USDT: Potential breakout or breakdown in progress!
USDC/USDT: Potential breakout or breakdown 

In [1]:
#WeakCryptos
import ccxt
import pandas as pd

# Initialize Binance API
exchange = ccxt.binance({
    'options': {'defaultType': 'future'},  # Enable Futures
})

# Fetch Historical Data for a Symbol
def fetch_data(symbol, timeframe='1d', limit=1500):
    try:
        ohlcv = exchange.fetch_ohlcv(symbol, timeframe, limit=limit)
        df = pd.DataFrame(ohlcv, columns=['timestamp', 'open', 'high', 'low', 'close', 'volume'])
        df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms')
        return df
    except (ccxt.BadSymbol, ccxt.NetworkError, ccxt.BaseError) as e:
        
        return None

# Calculate Metrics
def calculate_metrics(df):
    # Check if close < open
    df['close_below_open'] = df['close'] < df['open']

    # Fade % when close < open
    df['fade_percent'] = ((df['open'] - df['close']) / df['open']) * 100
    df.loc[~df['close_below_open'], 'fade_percent'] = None  # Only calculate for close < open

    # Low % from open when close < open
    df['low_from_open_percent'] = ((df['open'] - df['low']) / df['open']) * 100
    df.loc[~df['close_below_open'], 'low_from_open_percent'] = None

    # Metrics
    total_days = len(df)
    close_below_open_days = df['close_below_open'].sum()
    percentage_close_below_open = (close_below_open_days / total_days) * 100

    avg_fade = df['fade_percent'].mean()
    avg_low_from_open = df['low_from_open_percent'].mean()

    return {
        'total_days': total_days,
        'close_below_open_days': close_below_open_days,
        'percentage_close_below_open': percentage_close_below_open,
        'avg_fade_percent': avg_fade,
        'avg_low_from_open_percent': avg_low_from_open,
    }

# Get Available Symbols
def get_symbols():
    markets = exchange.load_markets()
    symbols = [symbol for symbol in markets if symbol.endswith('/USDT')]  # Filter USDT pairs
    
    return symbols

# Filter Symbols Based on Fade % > 7
def filter_symbols_by_fade():
    symbols = get_symbols()
    high_fade_symbols = []

    for symbol in symbols:
        df = fetch_data(symbol)
        if df is None:
            continue  # Skip invalid symbols

        metrics = calculate_metrics(df)
        
        # Check if avg_fade_percent is greater than 7
        if metrics['avg_fade_percent'] > 7:
            high_fade_symbols.append({
                'symbol': symbol,
                'avg_fade_percent': metrics['avg_fade_percent']
            })

    return high_fade_symbols

# Example Usage
high_fade_symbols = filter_symbols_by_fade()

# Display Results
print("Cryptos with Fade % greater than 7:")
for symbol_info in high_fade_symbols:
    print(f"Symbol: {symbol_info['symbol']}, Avg Fade %: {symbol_info['avg_fade_percent']:.2f}%")



##What’s Considered High?
#Context Matters Again:
#In most cases, a value above 5%-8% would indicate substantial intraday volatility or sharp sell-offs before some recovery by close.
#For highly volatile assets like cryptocurrencies, values above 10%-12% could be considered high.
#Significance of High Avg Low from Open:
#Intraday Volatility: Indicates significant bearish price action during the day, even if partial recovery occurred before close.
#Stop Runs or Liquidations: A sharp dip to a low might indicate stop-loss cascades or liquidations in leveraged markets.
#Reversal Opportunity: Large intraday dips followed by a partial recovery can sometimes signal exhaustion in the bearish trend, especially if accompanied by volume spikes.
#How to Define High Based on Metrics
#Historical Context:

#Use historical data for the symbol to calculate average and standard deviation for fade % and low-from-open %.
#Define "high" as exceeding the mean + 1.5 to 2 standard deviations.
#Example Thresholds for Cryptocurrencies (Volatile Markets):

#High Fade %: >7-10%
#High Avg Low %: >10-12%
#Example Thresholds for Stable Markets (e.g., Stock Indices):

#High Fade %: >2-5%
#High Avg Low %: >5-8%
#Practical Use
#Trading Implications:

#High Fade %: Look for continuation in downtrend or oversold conditions for potential bounce setups.
#High Avg Low %: Monitor for volatility spikes and potential reversal opportunities.


Cryptos with Fade % greater than 7:
Symbol: C/USDT, Avg Fade %: 7.11%
Symbol: SOMI/USDT, Avg Fade %: 9.68%
Symbol: OPEN/USDT, Avg Fade %: 9.94%
Symbol: LINEA/USDT, Avg Fade %: 8.31%
Symbol: HOLO/USDT, Avg Fade %: 10.61%
Symbol: PUMP/USDT, Avg Fade %: 7.21%
Symbol: AVNT/USDT, Avg Fade %: 10.26%
Symbol: ZKC/USDT, Avg Fade %: 11.66%
Symbol: BARD/USDT, Avg Fade %: 7.62%
Symbol: 0G/USDT, Avg Fade %: 8.10%
Symbol: HEMI/USDT, Avg Fade %: 7.33%
Symbol: XPL/USDT, Avg Fade %: 7.94%
Symbol: MIRA/USDT, Avg Fade %: 10.78%
Symbol: FF/USDT, Avg Fade %: 13.85%
Symbol: EDEN/USDT, Avg Fade %: 13.71%
Symbol: NOM/USDT, Avg Fade %: 7.18%
Symbol: 2Z/USDT, Avg Fade %: 10.54%
Symbol: ASTER/USDT, Avg Fade %: 9.12%
Symbol: EUL/USDT, Avg Fade %: 9.75%
Symbol: ENSO/USDT, Avg Fade %: 10.97%
Symbol: YB/USDT, Avg Fade %: 14.83%
Symbol: ZBT/USDT, Avg Fade %: 27.10%


In [9]:
from binance.client import Client
import pandas as pd
import numpy as np
from datetime import datetime, timezone

# === Initialize Binance Client ===
api_key = "your_api_key"
api_secret = "your_api_secret"
client = Client(api_key, api_secret)

# === Fetch USDT futures symbols ===
def fetch_usdt_futures_symbols():
    try:
        futures_info = client.futures_exchange_info()
        usdt_symbols = [
            symbol['symbol']
            for symbol in futures_info['symbols']
            if symbol['symbol'].endswith('USDT')
        ]
        return usdt_symbols
    except Exception as e:
        print(f"Error fetching USDT futures symbols: {e}")
        return []

# === Fetch historical data ===
def fetch_historical_klines(symbol, interval, lookback):
    try:
        klines = client.futures_klines(symbol=symbol, interval=interval, limit=lookback)
        df = pd.DataFrame(klines, columns=[
            'open_time', 'open', 'high', 'low', 'close', 'volume',
            'close_time', 'quote_asset_volume', 'number_of_trades',
            'taker_buy_base_asset_volume', 'taker_buy_quote_asset_volume', 'ignore'
        ])
        df[['open', 'high', 'low', 'close']] = df[['open', 'high', 'low', 'close']].astype(float)
        df['open_time'] = pd.to_datetime(df['open_time'], unit='ms', utc=True)
        df['close_time'] = pd.to_datetime(df['close_time'], unit='ms', utc=True)
        return df[['open_time', 'close_time', 'open', 'high', 'low', 'close']]
    except Exception as e:
        print(f"Error fetching historical data for {symbol}: {e}")
        return pd.DataFrame()

# === Detect strong rejection candles ===
def is_rejection_candle(open_price, high_price, low_price, close_price, avg_range,
                        rejection_threshold=0.65, close_zone=0.25):
    total_range = high_price - low_price
    if total_range == 0 or np.isnan(avg_range):
        return False, None

    upper_shadow = high_price - max(open_price, close_price)
    lower_shadow = min(open_price, close_price) - low_price

    # Candle must be larger than average
    if total_range < 1.5 * avg_range:
        return False, None

    upper_ratio = upper_shadow / total_range
    lower_ratio = lower_shadow / total_range
    close_position = (close_price - low_price) / total_range  # 0 = bottom, 1 = top

    # Bearish rejection (long upper wick, close near bottom)
    if upper_ratio > rejection_threshold and close_position < close_zone:
        return True, "bearish"

    # Bullish rejection (long lower wick, close near top)
    if lower_ratio > rejection_threshold and close_position > (1 - close_zone):
        return True, "bullish"

    return False, None

# === Find today's rejection candles ===
def find_todays_rejections(usdt_symbols, interval, lookback):
    results = []
    for symbol in usdt_symbols:
        data = fetch_historical_klines(symbol, interval, lookback)
        if data.empty:
            continue

        # Calculate average candle range for context
        data['range'] = data['high'] - data['low']
        data['avg_range'] = data['range'].rolling(20).mean()

        # Check only the latest completed candle
        last = data.iloc[-1]
        avg_range = data['avg_range'].iloc[-2] if len(data) > 2 else data['range'].mean()

        is_reject, reject_type = is_rejection_candle(
            last['open'], last['high'], last['low'], last['close'], avg_range
        )

        if is_reject:
            results.append({
                "symbol": symbol,
                "type": reject_type,
                "open": last['open'],
                "high": last['high'],
                "low": last['low'],
                "close": last['close'],
                "time": last['close_time']
            })
    return results

# === Run the radar ===
interval = '1d'  # daily timeframe
lookback = 50
usdt_futures_symbols = fetch_usdt_futures_symbols()
todays_rejections = find_todays_rejections(usdt_futures_symbols, interval, lookback)

# === Print results ===
print("\n🔥 Strong Rejection Candles Detected Today 🔥")
for result in todays_rejections:
    t = result['time'].strftime("%Y-%m-%d")
    print(f"{t} | {result['symbol']:<10} | {result['type'].upper():<7} | "
          f"O:{result['open']:.2f} H:{result['high']:.2f} L:{result['low']:.2f} C:{result['close']:.2f}")


Error fetching historical data for BTCSTUSDT: APIError(code=-1122): Invalid symbol status.

🔥 Strong Rejection Candles Detected Today 🔥


In [11]:
import ccxt
import pandas as pd
from datetime import datetime

# Initialize Binance
exchange = ccxt.binance()

# Get all USDT trading pairs
markets = exchange.load_markets()
symbols = [s for s in markets if s.endswith('/USDT') and not s.startswith('BUSD')]

bullish = []
bearish = []

for symbol in symbols:
    try:
        # Fetch the last 2 daily candles
        ohlcv = exchange.fetch_ohlcv(symbol, timeframe='1d', limit=2)
        if not ohlcv:
            continue
        
        ts, open_, high, low, close, vol = ohlcv[-1]  # most recent candle
        
        # Compute range position of close
        if high != low:
            close_pos = (close - low) / (high - low)
        else:
            continue
        
        # Bullish engulfing-like condition: close in top 25% of range
        if close_pos >= 0.75 and close > open_:
            bullish.append((symbol, close_pos, close, vol))
        
        # Bearish engulfing-like condition: close in bottom 25% of range
        if close_pos <= 0.25 and close < open_:
            bearish.append((symbol, close_pos, close, vol))
    
    except Exception as e:
        print(f"Error on {symbol}: {e}")

# Convert to DataFrame for nice viewing
bullish_df = pd.DataFrame(bullish, columns=['Symbol', 'ClosePos', 'Close', 'Volume']).sort_values('ClosePos', ascending=False)
bearish_df = pd.DataFrame(bearish, columns=['Symbol', 'ClosePos', 'Close', 'Volume']).sort_values('ClosePos')

print("\n=== Bullish Candidates (close in top 25%) ===")
print(bullish_df.head(20))

print("\n=== Bearish Candidates (close in bottom 25%) ===")
print(bearish_df.head(20))



=== Bullish Candidates (close in top 25%) ===
           Symbol  ClosePos        Close        Volume
166     USDE/USDT  1.000000     0.999300  1.366430e+07
135        G/USDT  1.000000     0.007410  5.347400e+06
138     EURI/USDT  1.000000     1.160300  2.019630e+04
5       TUSD/USDT  1.000000     0.996700  2.236400e+04
141      SCR/USDT  1.000000     0.164000  4.551574e+05
53       JST/USDT  1.000000     0.033380  4.801568e+06
147     MOVE/USDT  1.000000     0.075000  2.252840e+06
44       PNT/USDT  1.000000     0.035000  1.924063e+07
20     USDSB/USDT  1.000000     0.999700  2.989000e+01
165      A2Z/USDT  0.987805     0.003888  1.447856e+07
156      NIL/USDT  0.979310     0.307500  2.186815e+06
132      TAO/USDT  0.969388   386.600000  1.824227e+04
86      BOND/USDT  0.957845     2.152000  1.447110e+06
157  MUBARAK/USDT  0.955556     0.023470  9.301736e+06
11     WAVES/USDT  0.954198     1.076000  8.549053e+05
110      STG/USDT  0.952381     0.159500  1.040333e+06
1        BNB/USDT 